In [1]:
#Ishaan
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

df = pd.read_csv("data/nodules_labels.csv")
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]



In [2]:
#train; input: the size, as log-diameter rescaled to sit around 0
d = train["diameter_mm"].to_numpy()     # plain numbers, shape (1305,)
logd = np.log(d)
m = logd.mean()    
std = logd.std()    #standard deviation
x = (logd - m) / std
x_train = torch.tensor(x, dtype=torch.float32)[:, None]  # shape (1305, 1)
# target: the rating as a class number 0-4
y_train = torch.tensor(train["hard"].to_numpy() - 1, dtype=torch.int64)   # shape (1305,)

#val
d = val["diameter_mm"].to_numpy()
logd = np.log(d)
x = (logd-m)/std
x_val = torch.tensor(x,dtype=torch.float32)[:,None]
y_val = torch.tensor(val["hard"].to_numpy()-1, dtype=torch.int64)
print(x_val.shape)

torch.Size([290, 1])


In [3]:
model = nn.Linear(1,5)
loss_fn = nn.CrossEntropyLoss() #measures how wrong the model's 5 scores are compared with the true class.
optimiser = torch.optim.Adam(model.parameters(), lr=0.05) #adjusts the model's 10 numbers (5 weights and 5 biases) to make the loss smaller. learning rate=0.05 is how big each adjustment is.
for step in range(500):
    optimiser.zero_grad()           # 1. clear the previous step's gradients
    out = model(x_train)            # 2. forward: 5 scores per nodule, shape (1305, 5)
    loss = loss_fn(out, y_train)    # 3. how wrong is it?
    loss.backward()                 # 4. work out which way to nudge each number
    optimiser.step()                # 5. nudge them

    if step % 50 == 0:
        print(step, loss.item())

0 2.1727583408355713
50 1.1416919231414795
100 1.1281523704528809
150 1.1263467073440552
200 1.1260185241699219
250 1.125964879989624
300 1.1259578466415405
350 1.1259572505950928
400 1.1259571313858032
450 1.1259570121765137


In [4]:
import sys; sys.path.insert(0, "pipeline")
import metrics as M

with torch.no_grad():                                        # scoring only, no learning
    probs = torch.softmax(model(x_val), dim=1).numpy()      # (290, 5), each row sums to 1

hard = val["hard"].to_numpy()
soft = val[[f"soft_{k}" for k in range(1, 6)]].to_numpy()

print(M.five_class(probs, hard, soft))
print("AUC", M.roc_auc_score(M.suspicious(hard), M.p_suspicious(probs)))

{'accuracy': 0.5655172413793104, 'balanced_accuracy': 0.37426139088729016, 'macro_f1': 0.35114199726402184, 'quadratic_kappa': 0.46794985274474143, 'mae_rating': np.float64(0.5517241379310345), 'nll_vs_median': np.float32(1.1079032), 'ce_vs_readers': np.float64(1.4109660437309193), 'ece': np.float64(0.02243797224143451)}
AUC 0.9131006006006006
